- r=8
- all 7 LoRA targets: q/k/v/o/gate/up/down
- flat LR = 2e-4
- 3 epochs
- train + val combined
- letter-only target

### improvised:
- correct masking + right-padding during training
- Drive checkpoints
- safe image handling
- fixed inference setup
- submission validation checks

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" accelerate bitsandbytes "pandas==2.2.2" "pillow==11.3.0" tqdm kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.5 MB/s eta 0:00:00


In [2]:
import os, json, random, gc, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, PeftModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU: True
Tesla T4


In [3]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
assert len(matches) > 0

DATA_ROOT = matches[0].parents[2]

print("DATA_ROOT:", DATA_ROOT)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

100%|██████████| 358M/358M [00:01<00:00, 200MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions
DATA_ROOT: /root/.cache/kagglehub/competitions/pixels-to-predictions/images
Train: 3109 Val: 1048 Test: 1008


## Cell 3 - combine train + val

In [5]:
trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

print("Combined train+val:", len(trainval_df))
print("Test:", len(test_df))

Combined train+val: 4157
Test: 1008


In [6]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = Path("/content/lora_v9_r8_all7_flatlr_trainval")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval")

Mounted at /content/drive


In [7]:
IMAGE_CAP = 768   # if OOM, change to 512

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    if not path.exists():
        path = DATA_ROOT.parent / row["image_path"]

    img = Image.open(path).convert("RGB")
    img.thumbnail((IMAGE_CAP, IMAGE_CAP), Image.BICUBIC)
    return img


def img_384(row):
    path = DATA_ROOT / row["image_path"]
    if not path.exists():
        path = DATA_ROOT.parent / row["image_path"]

    img = Image.open(path).convert("RGB")
    img.thumbnail((384, 384), Image.BICUBIC)
    return img

In [8]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_prompt(row, include_answer=False):
    choices = row["choices"]

    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {choice}"
        for i, choice in enumerate(choices)
    )

    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    # 400-char truncation per strategy
    if lecture:
        lecture = lecture[:400]
    if hint:
        hint = hint[:400]

    prompt = "<image>\n"
    prompt += "Answer the multiple-choice question.\n\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"
    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}"

    return prompt


print(build_prompt(trainval_df.iloc[0], include_answer=True)[:1000])

<image>
Answer the multiple-choice question.

Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete 

Hint: Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, the female frog lays eggs on a plant. When tadpoles hatch from the eggs, the male frog lets the tadpoles climb onto hi

Question: Why might putting each tadpole in its own pool of water increase the reproductive success of a male Amazonian poison frog? Co

In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", trainable)
assert trainable <= 5_000_000

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339
Trainable params: 4784128


In [10]:
class VQATrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return {
            "image": load_image(row),
            "prompt_text": build_prompt(row, include_answer=False),
            "full_text": build_prompt(row, include_answer=True),
        }


def collate_train(batch):
    images = [x["image"] for x in batch]
    full_texts = [x["full_text"] for x in batch]
    prompt_texts = [x["prompt_text"] for x in batch]

    full_inputs = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    prompt_inputs = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    labels = full_inputs["input_ids"].clone()
    labels[:] = -100

    for i in range(len(batch)):
        full_len = int(full_inputs["attention_mask"][i].sum().item())
        prompt_len = int(prompt_inputs["attention_mask"][i].sum().item())
        labels[i, prompt_len:full_len] = full_inputs["input_ids"][i, prompt_len:full_len]

    full_inputs["labels"] = labels
    return full_inputs

In [11]:
processor.tokenizer.padding_side = "right"

tmp_ds = VQATrainDataset(trainval_df.head(4))
tmp_batch = collate_train([tmp_ds[i] for i in range(4)])

for i in range(4):
    labels = tmp_batch["labels"][i]
    visible = labels[labels != -100]
    print("Loss text:", repr(processor.tokenizer.decode(visible)))

Loss text: ' C'
Loss text: ' A'
Loss text: ' B'
Loss text: ' B'


In [12]:
SMOKE_STEPS = 200
GRAD_ACCUM = 8
LR = 2e-4

smoke_df = trainval_df.sample(frac=0.10, random_state=SEED).reset_index(drop=True)

smoke_ds = VQATrainDataset(smoke_df)
smoke_loader = DataLoader(
    smoke_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

losses = []
optimizer.zero_grad(set_to_none=True)

for step, batch in enumerate(tqdm(smoke_loader, desc="Smoke test")):
    if step >= SMOKE_STEPS:
        break

    batch = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    out = model(**batch)
    loss = out.loss / GRAD_ACCUM
    loss.backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    losses.append(out.loss.item())

    if (step + 1) % 32 == 0:
        print(f"Step {step+1}: avg32={np.mean(losses[-32:]):.4f}")

print("First 32 avg:", np.mean(losses[:32]))
print("Last 32 avg:", np.mean(losses[-32:]))

del optimizer, smoke_loader, smoke_ds
gc.collect()
torch.cuda.empty_cache()

Smoke test:   0%|          | 0/416 [00:00<?, ?it/s]

Step 32: avg32=0.7698
Step 64: avg32=0.9087
Step 96: avg32=1.0201
Step 128: avg32=0.8819
Step 160: avg32=0.8258
Step 192: avg32=0.6182
First 32 avg: 0.7697886526658522
Last 32 avg: 0.5582248946884647


In [13]:
del model, base_model
gc.collect()
torch.cuda.empty_cache()

processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

model = get_peft_model(base_model, lora_config)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", trainable)
assert trainable <= 5_000_000

trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339
Trainable params: 4784128


In [15]:
BATCH_SIZE_TRAIN = 1
GRAD_ACCUM = 8
EPOCHS = 3
LR = 2e-4
CHECKPOINT_EVERY = 500

processor.tokenizer.padding_side = "right"

train_ds = VQATrainDataset(trainval_df)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

loss_history = []
optimizer.zero_grad(set_to_none=True)
global_step = 0

for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(pbar):
        batch = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        loss_history.append(out.loss.item())
        global_step += 1

        if global_step % 50 == 0:
            pbar.set_postfix(
                loss=f"{loss_history[-1]:.4f}",
                avg50=f"{np.mean(loss_history[-50:]):.4f}",
                lr=f"{LR:.2e}",
            )

        if global_step % CHECKPOINT_EVERY == 0:
          recent_loss = float(np.mean(loss_history[-50:])) if len(loss_history) >= 50 else float(np.mean(loss_history))

          ckpt_dir = Path(f"/content/drive/MyDrive/lora_v9_ckpt_step_{global_step}")
          ckpt_dir.mkdir(parents=True, exist_ok=True)

          model.save_pretrained(ckpt_dir)
          processor.save_pretrained(ckpt_dir)

          print(f"\nSaved checkpoint: {ckpt_dir}")
          print(f"Step {global_step} | avg_last_50_loss={recent_loss:.4f} | epoch={epoch+1}/{EPOCHS}\n")

print("Training done")
print("First 50 avg:", np.mean(loss_history[:50]))
print("Last 50 avg:", np.mean(loss_history[-50:]))

Epoch 1/3:   0%|          | 0/4157 [00:00<?, ?it/s]


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_500
Step 500 | avg_last_50_loss=0.6720 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_1000
Step 1000 | avg_last_50_loss=0.7500 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_1500
Step 1500 | avg_last_50_loss=0.5262 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_2000
Step 2000 | avg_last_50_loss=0.4989 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_2500
Step 2500 | avg_last_50_loss=0.4238 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_3000
Step 3000 | avg_last_50_loss=0.3680 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_3500
Step 3500 | avg_last_50_loss=0.5828 | epoch=1/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_4000
Step 4000 | avg_last_50_loss=0.7017 | epoch=1/3



Epoch 2/3:   0%|          | 0/4157 [00:00<?, ?it/s]


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_4500
Step 4500 | avg_last_50_loss=0.3815 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_5000
Step 5000 | avg_last_50_loss=0.6236 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_5500
Step 5500 | avg_last_50_loss=0.3198 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_6000
Step 6000 | avg_last_50_loss=0.2570 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_6500
Step 6500 | avg_last_50_loss=0.4128 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_7000
Step 7000 | avg_last_50_loss=0.3726 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_7500
Step 7500 | avg_last_50_loss=0.3707 | epoch=2/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_8000
Step 8000 | avg_last_50_loss=0.1665 | epoch=2/3



Epoch 3/3:   0%|          | 0/4157 [00:00<?, ?it/s]


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_8500
Step 8500 | avg_last_50_loss=0.2648 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_9000
Step 9000 | avg_last_50_loss=0.4373 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_9500
Step 9500 | avg_last_50_loss=0.2351 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_10000
Step 10000 | avg_last_50_loss=0.0902 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_10500
Step 10500 | avg_last_50_loss=0.2051 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_11000
Step 11000 | avg_last_50_loss=0.0594 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_11500
Step 11500 | avg_last_50_loss=0.5595 | epoch=3/3


Saved checkpoint: /content/drive/MyDrive/lora_v9_ckpt_step_12000
Step 12000 | avg_last_50_loss=0.2547 | epoch=3/3

Training done
First 50 avg: 0.5564756438661425
Last 50 avg: 0.1353539931582827

In [16]:
model.eval()
model.config.use_cache = True

SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

if DRIVE_SAVE.exists():
    shutil.rmtree(DRIVE_SAVE)

shutil.copytree(SAVE_DIR, DRIVE_SAVE)

print("Saved to:", DRIVE_SAVE)
print(list(DRIVE_SAVE.iterdir()))

Saved to: /content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval
[PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/chat_template.jinja'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/preprocessor_config.json'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/README.md'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/merges.txt'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/adapter_model.safetensors'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/adapter_config.json'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/vocab.json'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/special_tokens_map.json'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/added_tokens.json'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval/processor_config.json'), PosixPath('/content/drive/MyDrive/lora_v9_r8_all7_flatlr_

In [17]:
processor.tokenizer.padding_side = "left"
model.eval()
model.config.use_cache = True

def get_candidate_token_ids(tokenizer):
    candidate_ids = {}

    for letter in CHOICE_LETTERS:
        forms = [letter, " " + letter, "\n" + letter, letter + ".", " " + letter + "."]
        ids = set()

        for form in forms:
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])

        candidate_ids[letter] = sorted(ids)

    return candidate_ids


candidate_ids = get_candidate_token_ids(processor.tokenizer)


def predict_score_one(row):
    image = img_384(row)
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    log_probs = torch.log_softmax(logits, dim=-1)

    scores = []
    for choice_idx in range(len(row["choices"])):
        letter = CHOICE_LETTERS[choice_idx]
        scores.append(log_probs[0, candidate_ids[letter]].max().item())

    return int(np.argmax(scores)), scores

In [18]:
val_preds = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Sanity val"):
    pred, _ = predict_score_one(row)
    val_preds.append(pred)
    torch.cuda.empty_cache()

sanity_acc = np.mean(np.array(val_preds) == val_df["answer"].values)
print("Sanity val accuracy:", sanity_acc)

Sanity val:   0%|          | 0/1048 [00:00<?, ?it/s]

Sanity val accuracy: 0.9141221374045801


In [ ]:
test_preds = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test"):
    pred, _ = predict_score_one(row)
    test_preds.append(pred)
    torch.cuda.empty_cache()

print("Predictions:", len(test_preds), "Test rows:", len(test_df))

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": test_preds,
})

submission["answer"] = submission["answer"].astype(int)

assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(sample_submission)
assert submission["id"].tolist() == sample_submission["id"].tolist()
assert submission["answer"].notna().all()

for idx, row in test_df.iterrows():
    pred = int(submission.loc[idx, "answer"])
    assert 0 <= pred < len(row["choices"])

submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_lora_v9_r8_all7_flatlr_trainval.csv", index=False)

print(submission.head())
print(submission["answer"].value_counts().sort_index())
print("Saved submission")

In [22]:
from google.colab import files
files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>